# 03 — Time-Delay Embedding (TDE)

Estimates embedding parameters `τ` (AMI) and `m` (FNN) for each trial, then aggregates them per participant by taking the median across the 3 conditions.

## Signal used

`signal_final` (100 Hz) with the **perturbation zone excluded**: concatenation of pre-perturbation `[start, entry − 5 s]` and post-perturbation `[exit + 30 s, end]`.

Reasons:
- Pre-perturbation alone yields ~95 cycles, below Mehdizadeh (2020) recommended minimum (200–300).
- Concatenation yields ~525–630 cycles → stable `m`.
- Characterises the participant's normal locomotion, not the perturbation transient.

## Parameters

| Parameter | Value | Source |
|---|---|---|
| `max_lag` | 100 (= 1 s at 100 Hz) | — |
| `max_dim` | 10 | — |
| `r_threshold` | 15 | Kennel et al. 1992 |
| `fnn_threshold` | 10% | Mehdizadeh 2020 |
| `τ` | first local minimum of AMI | Fraser & Swinney 1986 |
| `m` | first dim with FNN rate < 10% | Kennel 1992 |

## Aggregation

`τ_p = round(median(τ_sil, τ_tr, τ_bm))` and likewise `m_p`. Assumption: `τ` and `m` are subject-level properties (cadence, dynamic complexity), not condition-dependent.

## Inputs / Outputs

- **Input**  : `data/processed/*.npz`
- **Output** : `outputs/tables/03_tde_per_trial.csv`, `outputs/tables/03_tde_per_participant.csv`, `outputs/figures/03_ami_curves.png`, `outputs/figures/03_fnn_curves.png`, `outputs/figures/03_distributions.png`

In [ ]:
from resilience import paths, participants, config
from resilience.io import writer
from resilience.processing import detection, tde

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
%matplotlib inline

pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 200)

OUT_TABLES  = Path(paths.OUTPUTS_DIR) / 'tables'
OUT_FIGURES = Path(paths.OUTPUTS_DIR) / 'figures'
OUT_TABLES.mkdir(parents=True, exist_ok=True)
OUT_FIGURES.mkdir(parents=True, exist_ok=True)

## 1. Load trials and re-run bbox detection

We reload every non-excluded trial and re-run the bbox detection to recover `entry_t` / `exit_t` (raw reference, i.e. with `+ crop_start_s`).

`signal_final` is in the **post-crop** reference → we will subtract `crop_start_s` before masking.

In [ ]:
def load_trials_with_detection():
    """Load all non-excluded trials with signal_final and bbox entry/exit."""
    trials = {}
    for code, P in participants.PARTICIPANTS.items():
        for cond, trial in P.trials.items():
            if trial.excluded:
                continue
            try:
                d = writer.load_processed(code, cond)
            except FileNotFoundError:
                print(f"  ⚠️  No .npz for {code}/{cond}")
                continue

            # Sacrum XY for bbox
            dos_xy = np.stack([d[f'raw_Dos{i:02d}'][:, :2] for i in range(1, 5)], axis=0)
            sacrum_xy = np.nanmean(dos_xy, axis=0)
            for ax in range(2):
                col = sacrum_xy[:, ax]
                if np.isnan(col).any():
                    idx = np.arange(len(col))
                    valid = ~np.isnan(col)
                    if valid.sum() >= 2:
                        col[~valid] = np.interp(idx[~valid], idx[valid], col[valid])
                    sacrum_xy[:, ax] = col

            det = detection.detect_sand_by_bbox(sacrum_xy)
            offset = trial.crop_start_s if trial.crop_start_s is not None else 0.0
            entry_brut = det['entry_t'] + offset if not np.isnan(det['entry_t']) else np.nan
            exit_brut  = det['exit_t']  + offset if not np.isnan(det['exit_t'])  else np.nan

            trials[(code, cond)] = {
                'signal_final': d['signal_final'].astype(float),
                'time_final':   d['time_final'].astype(float),  # post-crop reference
                'crop_start_s': offset,
                'entry_brut':   entry_brut,
                'exit_brut':    exit_brut,
            }
    return trials

TRIALS = load_trials_with_detection()
print(f"✅ {len(TRIALS)} trials loaded")

## 2. Build the perturbation-excluded signal

For each trial: drop `[entry − 5 s, exit + 30 s]` (raw reference) and concatenate pre + post.

In [ ]:
MARGIN_PRE_S  = 5.0
MARGIN_POST_S = 30.0

def build_clean_signal(trial_data):
    """Concatenate pre- and post-perturbation segments for stationary TDE."""
    sig  = trial_data['signal_final']
    t    = trial_data['time_final']    # post-crop, starts at 0
    offs = trial_data['crop_start_s']

    if np.isnan(trial_data['entry_brut']):
        return sig.copy(), len(sig), 0

    # Convert raw timestamps to post-crop reference
    entry_local = trial_data['entry_brut'] - offs
    exit_local  = trial_data['exit_brut']  - offs

    mask_pre  = t < (entry_local - MARGIN_PRE_S)
    mask_post = t > (exit_local  + MARGIN_POST_S)

    sig_clean = np.concatenate([sig[mask_pre], sig[mask_post]])
    n_excluded = int(((~mask_pre) & (~mask_post)).sum())
    return sig_clean, len(sig_clean), n_excluded


test_key = list(TRIALS.keys())[0]
sig_clean, n_kept, n_excl = build_clean_signal(TRIALS[test_key])
print(f"Test {test_key}: original  = {len(TRIALS[test_key]['signal_final'])} samples")
print(f"            clean     = {n_kept} samples ({n_kept/100:.1f} s)")
print(f"            excluded  = {n_excl} samples ({n_excl/100:.1f} s)")

## 3. Compute AMI + FNN for all trials

In [ ]:
MAX_LAG       = 100
MAX_DIM       = 10
FNN_THRESHOLD = 10.0   # %   (Mehdizadeh 2020; overrides 1% from config.py)
FNN_R         = 15.0   #     (Kennel 1992)

results = []
tde_curves = {}  # for plots

for (code, cond), data in TRIALS.items():
    sig_clean, n_kept, n_excl = build_clean_signal(data)
    duration_s = n_kept / config.FS_CLEAN
    n_cycles_estim = int(duration_s * 1.75)  # rough estimate, mean cadence ~1.75 Hz

    if n_kept < 200 * 100 / 1.75:
        print(f"  ⚠️  {code}/{cond}: only {n_cycles_estim} cycles (<200)")

    tau, ami_values = tde.compute_ami(sig_clean, max_lag=MAX_LAG)
    m, fnn_rates    = tde.compute_fnn(sig_clean, max_dim=MAX_DIM, tau=tau,
                                       threshold=FNN_THRESHOLD, r_threshold=FNN_R)

    results.append({
        'participant':   code,
        'condition':     cond,
        'n_samples':     n_kept,
        'duration_s':    round(duration_s, 1),
        'n_cycles_est':  n_cycles_estim,
        'tau':           tau,
        'm':             m,
    })
    tde_curves[(code, cond)] = {'ami': ami_values, 'fnn': fnn_rates, 'tau': tau, 'm': m}
    print(f"  {code}/{cond:22s}: τ={tau:3d}, m={m}, n_cycles≈{n_cycles_estim}")

df_tde = pd.DataFrame(results)
print(f"\n✅ TDE computed on {len(df_tde)} trials")
df_tde

## 4. Aggregate per participant

In [ ]:
df_p = df_tde.groupby('participant').agg(
    n_trials = ('tau', 'count'),
    tau_min  = ('tau', 'min'),
    tau_med  = ('tau', 'median'),
    tau_max  = ('tau', 'max'),
    m_min    = ('m',   'min'),
    m_med    = ('m',   'median'),
    m_max    = ('m',   'max'),
).reset_index()

df_p['tau_p']     = df_p['tau_med'].round().astype(int)
df_p['m_p']       = df_p['m_med'].round().astype(int)
df_p['tau_range'] = df_p['tau_max'] - df_p['tau_min']
df_p['m_range']   = df_p['m_max']   - df_p['m_min']

print("TDE parameters aggregated per participant (median across 3 conditions):")
df_p[['participant', 'n_trials', 'tau_min', 'tau_med', 'tau_max', 'tau_range',
      'm_min', 'm_med', 'm_max', 'm_range', 'tau_p', 'm_p']]

In [ ]:
print("Global distribution:\n")
print(df_tde[['tau', 'm', 'duration_s', 'n_cycles_est']].describe().round(1))

print(f"\nInter-condition variability:")
print(f"  tau_range > 5 samples : {(df_p['tau_range'] > 5).sum()} / {len(df_p)} participants")
print(f"  m_range  > 1          : {(df_p['m_range'] > 1).sum()} / {len(df_p)} participants")

## 5. AMI + FNN curves per participant

1 figure = 1 participant, with the 3 conditions overlaid.

In [ ]:
COND_COLORS = {
    'silence':            'steelblue',
    'tempo_random':       'darkorange',
    'beatmove_adaptatif': 'forestgreen',
}

by_p = {}
for (code, cond), curves in tde_curves.items():
    by_p.setdefault(code, {})[cond] = curves

n_p    = len(by_p)
n_cols = 3
n_rows = (n_p + n_cols - 1) // n_cols

# AMI curves
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 3 * n_rows))
for i, (code, curves) in enumerate(sorted(by_p.items())):
    ax = axes.flat[i]
    for cond, c in curves.items():
        ax.plot(range(1, len(c['ami']) + 1), c['ami'],
                color=COND_COLORS[cond], linewidth=1, alpha=0.8,
                label=f"{cond[:3]} τ={c['tau']}")
        ax.axvline(c['tau'], color=COND_COLORS[cond], linestyle=':', linewidth=0.8, alpha=0.5)
    ax.set_title(code, fontsize=9)
    ax.set_xlabel('Lag', fontsize=8)
    ax.set_ylabel('AMI', fontsize=8)
    ax.legend(fontsize=6, loc='upper right')
    ax.grid(alpha=0.3)
    ax.tick_params(labelsize=7)
for j in range(i + 1, n_rows * n_cols):
    axes.flat[j].axis('off')
plt.suptitle('AMI vs lag (3 conditions per participant) — τ = first local minimum',
             fontweight='bold')
plt.tight_layout()
plt.savefig(OUT_FIGURES / '03_ami_curves.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# FNN curves
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 3 * n_rows))
for i, (code, curves) in enumerate(sorted(by_p.items())):
    ax = axes.flat[i]
    for cond, c in curves.items():
        ax.plot(range(1, len(c['fnn']) + 1), c['fnn'],
                color=COND_COLORS[cond], linewidth=1, alpha=0.8,
                marker='o', markersize=3,
                label=f"{cond[:3]} m={c['m']}")
        ax.axvline(c['m'], color=COND_COLORS[cond], linestyle=':', linewidth=0.8, alpha=0.5)
    ax.axhline(FNN_THRESHOLD, color='red', linestyle='--', linewidth=0.6, alpha=0.4,
               label=f'{FNN_THRESHOLD}%')
    ax.set_title(code, fontsize=9)
    ax.set_xlabel('Embedding dim', fontsize=8)
    ax.set_ylabel('FNN (%)', fontsize=8)
    ax.set_yscale('log')
    ax.legend(fontsize=6, loc='upper right')
    ax.grid(alpha=0.3, which='both')
    ax.tick_params(labelsize=7)
for j in range(i + 1, n_rows * n_cols):
    axes.flat[j].axis('off')
plt.suptitle(f'FNN rate vs dim (3 conditions per participant) — m = first dim below {FNN_THRESHOLD}%',
             fontweight='bold')
plt.tight_layout()
plt.savefig(OUT_FIGURES / '03_fnn_curves.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Global τ / m distributions
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df_tde['tau'], bins=range(df_tde['tau'].min(), df_tde['tau'].max() + 2),
             color='steelblue', edgecolor='black', alpha=0.7)
axes[0].axvline(df_tde['tau'].median(), color='red', linestyle='--',
                label=f"median = {df_tde['tau'].median():.0f}")
axes[0].set_xlabel('τ (samples)')
axes[0].set_ylabel('Trial count')
axes[0].set_title(f'τ distribution across {len(df_tde)} trials')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].hist(df_tde['m'], bins=range(df_tde['m'].min(), df_tde['m'].max() + 2),
             color='darkorange', edgecolor='black', alpha=0.7)
axes[1].axvline(df_tde['m'].median(), color='red', linestyle='--',
                label=f"median = {df_tde['m'].median():.0f}")
axes[1].set_xlabel('m (embedding dim)')
axes[1].set_ylabel('Trial count')
axes[1].set_title(f'm distribution across {len(df_tde)} trials')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(OUT_FIGURES / '03_distributions.png', dpi=120, bbox_inches='tight')
plt.show()

## 6. Export

- `03_tde_per_trial.csv` — per-trial parameters (for debug or per-trial alternative)
- `03_tde_per_participant.csv` — aggregated parameters (consumed by `04_resilience.ipynb`)

In [ ]:
df_tde.to_csv(OUT_TABLES / '03_tde_per_trial.csv', index=False)
df_p[['participant', 'tau_p', 'm_p', 'tau_range', 'm_range']].to_csv(
    OUT_TABLES / '03_tde_per_participant.csv', index=False)

print(f"✅ Exported:")
print(f"   {OUT_TABLES / '03_tde_per_trial.csv'}")
print(f"   {OUT_TABLES / '03_tde_per_participant.csv'}")